## Classes

In [ ]:
# Class vs instance attributes
# Instance attributes live on each object. Class attributes are shared. The classic bug: mutable class attributes shared by accident.

class BankAccount:
    interest_rate = 0.05 # class attr - shared by all instances

    def __init__(self, owner:str, balance: float = 0):
        self.owner = owner # instance attr - unique to each instance
        self._balance = balance # instance attr - unique to each instance
    
    @classmethod
    def from_dict(cls, data: dict):
        return cls(owner=data['owner'], balance=data.get('balance', 0))
    
    @staticmethod
    def validate_amount(amount: float) -> bool:
        return amount > 0 

In [ ]:
#The mutable default gotcha
#Mutable class attributes (lists, dicts) are shared — one of the most common intermediate-level bugs.

class BadCard:
    items = [] # # DANGER: shared across all instances

    def add(self,item): 
        self.items.append(item)

class GoodCart:
    def __init__(self):
        self.items = [] # instance attr - unique to each instance

    def add(self, item):
        self.items.append(item)

In [3]:
c1 , c2 = BadCard(), BadCard()
c1.add('apple')
print(c1.items) # ['apple']
print(c2.items) # ['apple'] - unexpected! shared across instances

['apple']
['apple']


Great question — this trips up a lot of intermediate devs. Here's the core difference:

| | `@classmethod` | `@staticmethod` |
|---|---|---|
| First param | `cls` (the class itself) | nothing special |
| Knows about the class? | Yes | No |
| Knows about instances? | No | No |
| Can be inherited? | Yes, `cls` adapts | Yes, but no class awareness |
| Main use | Alternative constructors, factory methods | Utility/helper functions |

**Simple rule of thumb:**
- Need `cls` to create or inspect the class → `@classmethod`
- Just a helper function that happens to belong here → `@staticmethod`
- Need `self` to access instance data → regular method

In [ ]:
#@staticmethod — just a regular function that lives inside a class for organisation. It doesn't know what class it's on.
class MathUtils:
    @staticmethod
    def add(a, b):
        return a + b   # no self, no cls — pure function

MathUtils.add(3, 4)    # 7

7

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name 

    @classmethod
    def create(cls, name):      # cls = whatever class called this
        return cls(name)        # creates an instance of THAT class   

class Dog(Animal):
    def speak(self):
        return f"Woof! I'm {self.name}"

d = Dog.create("Rex")           # cls is Dog, so returns a Dog
print(type(d))                  # <class 'Dog'>
print(d.speak())                # Woof! I'm Rex

#If you used @staticmethod here and hardcoded Animal(name), Dog.create() would return an Animal, not a Dog — breaking inheritance.


<class '__main__.Dog'>
Woof! I'm Rex


In [6]:
# The most common real-world use: alternative constructors
from datetime import date

class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    @classmethod
    def from_birth_year(cls, name, birth_year):   # construct from different data
        age = date.today().year - birth_year
        return cls(name, age)

    @classmethod
    def from_dict(cls, data: dict):               # construct from a dict
        return cls(data["name"], data["age"])

    @staticmethod
    def is_adult(age: int) -> bool:               # utility — doesn't need class
        return age >= 18

p1 = Person("Priya", 28)
p2 = Person.from_birth_year("Raj", 1995)
p3 = Person.from_dict({"name": "Sam", "age": 22})

print(Person.is_adult(16))   # False

False


## INheritance, Polymorphism

In [ ]:
#Inheritance & `super()`
# Use inheritance for is-a relationships. `super()` lets you extend, not replace, parent behaviour. 
# Python resolves method lookup via MRO (Method Resolution Order).


In [ ]:
class Animal:
    def __init__(self, name: str):
        self.name = name
    def speak(self): raise NotImplementedError

class Dog(Animal):
    def __init__(self, name: str, breed: str):
        super().__init__(name)   # chain to parent
        self.breed = breed
    def speak(self): return f"Woof! I'm {self.name}, a {self.breed}"

class Cat(Animal):
    ## init method will be inherited from Animal, no need to redefine
    def speak(self): return f"Meow! I'm {self.name}"

In [2]:
for a in [Dog("Rex", "Lab"), Cat("Luna")]:
    print(a.speak())

Woof! I'm Rex, a Lab
Meow! I'm Luna


In [4]:
# Multiple inheritance & MRO
# Python uses C3 linearisation to resolve which method gets called first. Inspect it with `ClassName.__mro__`.

class Flyable:
    def move(self): return "flying"

class Swimmable:
    def move(self): return "swimming"

class Duck(Flyable, Swimmable): pass

In [5]:
print(Duck().move())   # "flying" — Flyable first

flying


In [6]:
print(Duck.__mro__)    # (Duck, Flyable, Swimmable, object)

(<class '__main__.Duck'>, <class '__main__.Flyable'>, <class '__main__.Swimmable'>, <class 'object'>)


## Dunders

In [7]:
# Magic Methods , Operator overloading
#The most useful dunder methods
#Dunders make your objects behave like built-in Python types — supporting `+`, `len()`, comparison, `repr`, and iteration.

In [8]:
class Vector:
    def __init__(self, x, y): self.x, self.y = x, y

    def __repr__(self):            # REPL / repr()
        return f"Vector({self.x}, {self.y})"

    def __str__(self):             # print()
        return f"({self.x}, {self.y})"

    def __add__(self, other):      # v1 + v2
        return Vector(self.x+other.x, self.y+other.y)

    def __len__(self):             # len(v)
        return int((self.x**2+self.y**2)**0.5)

    def __eq__(self, other):       # v1 == v2
        return self.x==other.x and self.y==other.y

In [10]:
v1 = Vector(1, 2)
v2 = Vector(3, 4)

In [13]:
print(v1)          # (1, 2)
print(v2)

(1, 2)
(3, 4)


In [15]:
print(len(v1))    
print(len(v2))    

2
5


In [16]:
print(v1 + v2)

(4, 6)


In [17]:
print(repr(v1))   # Vector(1, 2)

Vector(1, 2)


`__slots__` for memory efficiency
By default each instance stores its attributes in a `__dict__`. `__slots__` removes it, saving 30–50% memory when creating many objects.

In [18]:
class Point:
    __slots__ = ("x", "y")
    def __init__(self, x, y): self.x, self.y = x, y

# No __dict__ — can't add arbitrary attributes

## @Property

In [ ]:
# Descriptors & properties, Encapsulation

* `@property` — controlled attribute access
* `@property` lets you expose an attribute while hiding validation or computation behind it. Far cleaner than Java-style getters/setters.

In [19]:
class Temperature:
    def __init__(self, celsius: float):
        self._celsius = celsius

    @property
    def celsius(self) -> float:
        return self._celsius

    @celsius.setter
    def celsius(self, value: float):
        if value < -273.15:
            raise ValueError("Below absolute zero!")
        self._celsius = value

    @property
    def fahrenheit(self) -> float:   # computed, read-only
        return self._celsius * 9/5 + 32

In [21]:
t = Temperature(100)
print(t.fahrenheit)

212.0


In [23]:
#t.celsius = -300 ## raises ValueError

## 5. DataClasse

In [24]:
# Python 3.7+ has `dataclasses` for boilerplate-free classes 
# Great for simple data containers.

* Dataclasses — OOP without the ceremony
* `@dataclass` auto-generates `__init__`, `__repr__`, and `__eq__`. Use `field(default_factory=...)` for mutable defaults to avoid the shared-list bug.

In [28]:
from dataclasses import dataclass , field

@dataclass
class Book:
    title: str 
    author: str 
    pages: int 
    tags: list[str] = field(default_factory=list)  # mutable default handled safely


@dataclass(frozen=True)   # immutable
class Point:
    x: float
    y: float

@dataclass(order=True)    # adds __lt__, __gt__ etc.
class Student:
    gpa: float
    name: str

In [29]:
b = Book("Fluent Python", "Luciano", 790)
print(b)  # Book(title='Fluent Python', ...)

Book(title='Fluent Python', author='Luciano', pages=790, tags=[])


In [31]:
students = [Student(4.5,"Alice"), Student(3.9,"Bob")]
print(sorted(students))  # sorted by gpa

[Student(gpa=3.9, name='Bob'), Student(gpa=4.5, name='Alice')]


In [ ]:
## Read the complete notebook and take each and every concepts ,explain each with more examples in another notebook.
## finally, summarise the complete notebook in 5 lines.
# create one more notebook on the same topic with problems and excercises for practice.